# Lesson 2: Deterministic and stochastic

**You will:** simulate the same decay two ways, as a smooth ODE and as random events molecule by molecule, and see what the ODE is an average of.

**The model** ([`decay.bngl`](decay.bngl)): `X -> nothing`, each molecule lasting a random time with rate `k = 0.1` per second. With only 20 molecules, chance matters.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

import bngsim

## The ODE

The ODE treats X as a continuous amount: `dX/dt = -k X`, so `X(t) = 20 e^(-k t)` exactly.

In [ ]:
ode = bngsim.Simulator(bngsim.Model.load("decay.bngl"), method="ode").run(t_span=(0, 30), n_points=31)
t, X_ode = ode.time, ode.observables["X"]
print(f"X at t = 10: exact {20 * math.exp(-1.0):.4f}, simulated {X_ode[10]:.4f}")

## The stochastic simulation

The SSA (Gillespie's algorithm) picks *when* the next molecule decays at random. Each run, with its own seed, is one possible history: whole molecules, falling one at a time.

In [ ]:
plt.plot(t, X_ode, "k", linewidth=2, label="ODE")
for seed in (1, 2, 3, 4, 5):
    ssa = bngsim.Simulator(bngsim.Model.load("decay.bngl"), method="ssa").run(t_span=(0, 30), n_points=301, seed=seed)
    plt.step(ssa.time, ssa.observables["X"], where="post", alpha=0.7, label=f"SSA, seed {seed}")
plt.xlabel("time (s)")
plt.ylabel("molecules of X")
plt.legend()
plt.show()

## The ODE is the average

Run 500 stochastic histories and average them. The mean follows the ODE, and the spread around it is what probability predicts: each molecule survives to time t with chance `p = e^(-k t)`, so the count is binomial, with variance `20 p (1 - p)`.

In [ ]:
reps = bngsim.Simulator(bngsim.Model.load("decay.bngl"), method="ssa").run_replicates(500, t_span=(0, 30), n_points=31, seed=1)
counts = np.array([r.observables["X"] for r in reps])
mean, sd = counts.mean(axis=0), counts.std(axis=0)

plt.plot(t, X_ode, "k", label="ODE")
plt.plot(t, mean, "o", label="mean of 500 SSA runs")
plt.fill_between(t, mean - sd, mean + sd, alpha=0.3, label="± one standard deviation")
plt.xlabel("time (s)")
plt.ylabel("molecules of X")
plt.legend()
plt.show()

p = math.exp(-1.0)
print(f"mean at t = 10: exact {20 * p:.2f}, 500 runs {mean[10]:.2f}")
print(f"variance at t = 10: exact {20 * p * (1 - p):.2f}, 500 runs {counts[:, 10].var():.2f}")
# 500 runs pin the mean to within a few standard errors of the exact value.
standard_error = math.sqrt(20 * p * (1 - p) / 500)
assert abs(mean[10] - 20 * p) < 4 * standard_error

## Try it

1. Set `X0` to 2000 (with `model.set_param("X0", 2000)` and `model.reset()`, as in lesson 1). Is the noise, relative to the amount, bigger or smaller? Why do ODEs work well for large numbers of molecules?
2. Inside a cell, which would you trust for 3 copies of a gene: the ODE or the SSA?